# 01 — Pipeline: Build All Datasets

Run this notebook top-to-bottom to (re)produce every file in `data/raw/`, `data/interim/`, and `data/processed/`.
Each section is independently re-runnable: if the output file already exists, the heavy computation
is skipped and the existing file is loaded instead.

**Sections:**
| # | What it does | Skip if … |
|---|---|---|
| 0 | BigQuery extraction (dry-run estimate + download) | `data/raw/` CSVs already exist |
| 1 | Preprocess raw extract → per-article parquet | `data/interim/gdelt_preprocessed.parquet` exists |
| 2–7 | Build all `data/processed/` aggregates | (always recomputes from `df` in memory) |

**First-time setup:** run `gcloud auth application-default login`. Project ID is read from `BIGQUERY_PROJECT` env var (set in `.env`).

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT      = Path("..").resolve()
RAW       = ROOT / "data" / "raw"
INTERIM   = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

# Make src.* importable from any cell regardless of which cells were run first
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

def _skip(path: Path) -> bool:
    """Return True (and print a notice) when path already exists."""
    if path.exists():
        print(f"  ↳ exists, skipping: {path.relative_to(ROOT)}")
        return True
    return False

## Section 0 — BigQuery extraction

Run these cells to (re)download the raw data from GDELT BigQuery.
**Skip this section entirely if `data/raw/` files already exist.**

### Prerequisites
- GCP credentials: `gcloud auth application-default login`
- Your GCP project ID set in the cell below (must have BigQuery API enabled)
- GDELT is a public dataset — no special access required, but scans count against your quota

### Cost estimate
The dry-run cells print estimated bytes scanned and cost **before** committing any query.
GDELT GKG is large; the governance filter typically reduces the scan to 5–15 GB.
BigQuery free tier: 1 TB/month. Beyond that: ~$5/TB.

In [2]:
import os
from google.cloud import bigquery

GCP_PROJECT = os.environ.get("BIGQUERY_PROJECT", "genai-gdelt")
print(f"GCP project: {GCP_PROJECT}")

client = bigquery.Client(project=GCP_PROJECT)

def _dry_run(sql_path: Path) -> int:
    """Return estimated bytes for a query without running it."""
    sql = sql_path.read_text()
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    return job.total_bytes_processed

QUERIES = ROOT / "queries"

for name in ("extract_genai_gov.sql", "count_genai_total.sql"):
    path = QUERIES / name
    try:
        b = _dry_run(path)
        gb = b / 1e9
        cost = b / 1e12 * 5
        print(f"\n{name}")
        print(f"  Estimated scan: {gb:.2f} GB  (~${cost:.4f} at $5/TB)")
    except Exception as e:
        print(f"{name}: dry-run failed — {e}")

GCP project: genai-gdelt-499513

extract_genai_gov.sql
  Estimated scan: 872.77 GB  (~$4.3639 at $5/TB)

count_genai_total.sql
  Estimated scan: 115.59 GB  (~$0.5780 at $5/TB)


### 0a — Download main corpus (`extract_genai_gov.sql` → `data/raw/gdelt_genai_gov.csv`)

Large query (~1.1 M rows). Uses the BigQuery Storage Read API via `create_read_session`
for fast columnar download. Falls back to standard `to_dataframe()` if the storage API
is unavailable. Skips if the file already exists.

In [3]:
CORPUS_CSV = RAW / "gdelt_genai_gov.csv"

if not _skip(CORPUS_CSV):
    print("Running extract_genai_gov.sql …  (this may take several minutes)")
    sql = (QUERIES / "extract_genai_gov.sql").read_text()

    try:
        # Fast path: Storage Read API (requires bigquery-storage extra)
        df_raw = client.query(sql).to_dataframe(create_bqstorage_client=True)
    except Exception:
        # Fallback: standard REST download
        df_raw = client.query(sql).to_dataframe()

    print(f"  {len(df_raw):,} rows downloaded")
    RAW.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(CORPUS_CSV, index=False)
    print(f"  Saved → {CORPUS_CSV.relative_to(ROOT)}")
else:
    print("  Skipped — delete the file to re-download")

  ↳ exists, skipping: data/raw/gdelt_genai_gov.csv
  Skipped — delete the file to re-download


### 0b — Download coverage counts (`count_genai_total.sql` → `data/raw/monthly_genai_total.csv`)

Small query (~50 rows, a few KB). Always fast.

In [4]:
TOTAL_CSV = RAW / "monthly_genai_total.csv"

if not _skip(TOTAL_CSV):
    print("Running count_genai_total.sql …")
    sql = (QUERIES / "count_genai_total.sql").read_text()
    df_total = client.query(sql).to_dataframe()
    df_total.to_csv(TOTAL_CSV, index=False)
    print(f"  {len(df_total)} months downloaded → {TOTAL_CSV.relative_to(ROOT)}")
    df_total
else:
    print("  Skipped — delete the file to re-download")

Running count_genai_total.sql …
  44 months downloaded → data/raw/monthly_genai_total.csv


## Section 1 — Preprocess raw GDELT extract

Reads `data/raw/gdelt_genai_gov.csv` (~4.4 GB, ~1.1 M rows) and runs the full preprocessing
pipeline: deduplication, date parsing, tone parsing, region extraction, frame flag assignment,
and event-window tagging.

Output: `data/interim/gdelt_preprocessed.parquet`

> **Runtime:** ~3–5 min on a laptop. Skip this cell if the parquet already exists.

In [4]:
from src.preprocessing import load_raw, run_pipeline

PREPROCESSED = INTERIM / "gdelt_preprocessed.parquet"

if not _skip(PREPROCESSED):
    print("Loading raw extract …")
    raw = load_raw(RAW / "gdelt_genai_gov.csv")
    print(f"  {len(raw):,} rows loaded")

    print("Running preprocessing pipeline …")
    df = run_pipeline(raw)
    print(f"  {len(df):,} rows after deduplication")

    df.to_parquet(PREPROCESSED, index=False)
    print(f"  Saved → {PREPROCESSED.relative_to(ROOT)}")
else:
    df = pd.read_parquet(PREPROCESSED)

print(f"\nDataset: {len(df):,} articles, {df['month'].nunique()} months")
df.head(2)

Loading raw extract …
  1,116,937 rows loaded
Running preprocessing pipeline …
  1,116,091 rows after deduplication
  Saved → data/interim/gdelt_preprocessed.parquet

Dataset: 1,116,091 articles, 44 months


,DATE,SourceCommonName,DocumentIdentifier,V2Themes,V2Tone,V2Locations,AllNames,Quotations,year,month,...,region,frame_innovation_opportunity,frame_risk_safety,frame_regulation_governance,frame_rights_privacy,frame_economic_competition_labour,frame_misinformation_integrity,dominant_frame,event_name,rel_month
0,2024-09-16,wacotrib.com,https://wacotrib.com/ap/national/5-things-to-k...,"EPU_POLICY_CONGRESSIONAL,7769;GENERAL_GOVERNME...","-3.756201275691,1.91353649893692,5.66973777462...","3#Washington, Washington, United States#US#USD...","former president donald trump,104;ryan wesley ...",1845|39||i am going to fight and die for ukrai...,2024,2024-09,...,US,0,0,1,0,0,0,regulation_governance,eu_ai_act_in_force,1
1,2024-11-01,wobm.com,https://wobm.com/ixp/385/p/mtv-jersey-shore-vi...,"MEDIA_MSM,625;TAX_FNCACT_SPEAKER,2339;SCIENCE,...","2.90178571428571,4.91071428571429,2.0089285714...","3#Staten Island, New York, United States#US#US...","donald trump,191;madison square garden,214;new...",114|31||jersey shore : family vacation#1123|31...,2024,2024-11,...,US,0,0,0,0,0,0,NaN,eu_ai_act_in_force,3


## Section 2 — Monthly volume and frame shares

Frame prevalence rates = fraction of that month's articles matching each frame.
Frames are multi-label (one article can match multiple), so column values can sum >1.

In [5]:
from src.analysis import monthly_volume, frame_shares

VOL_PATH    = PROCESSED / "monthly_volume.parquet"
FRAMES_PATH = PROCESSED / "monthly_frames.parquet"

vol    = monthly_volume(df)
shares = frame_shares(df)

vol.to_frame().to_parquet(VOL_PATH)
shares.to_parquet(FRAMES_PATH)

print(f"Monthly volume: {len(vol)} months, {vol.sum():,} total articles")
print(f"Frame shares (first 3 months):")
shares.head(3)

Monthly volume: 44 months, 1,116,091 total articles
Frame shares (first 3 months):


,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economic_competition_labour,misinformation_integrity
month,,,,,,
2022-11,0.155273,0.160951,0.585732,0.036317,0.140769,0.146728
2022-12,0.249195,0.186276,0.588074,0.057486,0.183190,0.051784
2023-01,0.282407,0.251940,1.000890,0.063732,0.422593,0.056354


## Section 3 — Event study data

For each of the 7 milestones, compute volume and frame shares in the ±3-month window.
Stacked into long-form for easy faceting.

In [6]:
from src.analysis import event_study
from src.dictionaries import MILESTONES

EVENT_PATH = PROCESSED / "event_studies.parquet"

records = []
for m in MILESTONES:
    result = event_study(df, m["name"])
    shares_m = result["shares"].copy()
    shares_m["volume"] = result["volume"]
    shares_m["milestone"] = m["name"]
    shares_m["milestone_date"] = m["date"]
    shares_m["rel_month"] = shares_m.index
    records.append(shares_m.reset_index(drop=True))

event_df = pd.concat(records, ignore_index=True)
event_df.to_parquet(EVENT_PATH, index=False)

print(f"Event studies: {event_df['milestone'].nunique()} milestones × 7 rel-months = {len(event_df)} rows")
event_df[event_df["milestone"] == "chatgpt_launch"]

Event studies: 7 milestones × 7 rel-months = 49 rows


,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economic_competition_labour,misinformation_integrity,volume,milestone,milestone_date,rel_month
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,chatgpt_launch,2022-11-30,-3
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,chatgpt_launch,2022-11-30,-2
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,chatgpt_launch,2022-11-30,-1
3,0.155273,0.160951,0.585732,0.036317,0.140769,0.146728,17788,chatgpt_launch,2022-11-30,0
4,0.249195,0.186276,0.588074,0.057486,0.183190,0.051784,14908,chatgpt_launch,2022-11-30,1
5,0.282407,0.251940,1.000890,0.063732,0.422593,0.056354,15722,chatgpt_launch,2022-11-30,2
6,0.352880,0.319099,2.133522,0.081836,0.436808,0.084667,16252,chatgpt_launch,2022-11-30,3


## Section 4 — Regional breakdowns

Pooled frame prevalence rates per region (US, EU, UK), plus quarterly panel for time-series.

In [7]:
from src.analysis import region_comparison, region_frame_shares_over_time

REG_PATH = PROCESSED / "regional_frames.parquet"
REG_Q_PATH = PROCESSED / "regional_frames_quarterly.parquet"

reg = region_comparison(df)
reg.to_parquet(REG_PATH)

reg_q = region_frame_shares_over_time(df, freq="Q")
reg_q.to_parquet(REG_Q_PATH, index=False)

print("Pooled frame shares by region:")
reg

Pooled frame shares by region:


,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economic_competition_labour,misinformation_integrity
region,,,,,,
EU,0.154275,0.438270,0.604669,0.072715,0.193172,0.074155
UK,0.245388,0.478737,0.844684,0.113096,0.217930,0.059563
US,0.285319,0.310379,1.096852,0.072775,0.247023,0.090412


## Section 5 — Coverage validation

Join monthly article counts from the governance corpus (`extract_genai_gov.sql`) with
total monthly GenAI counts (`count_genai_total.sql`) to compute the governance coverage rate.

> **Note:** `monthly_genai_total.csv` must be the BigQuery export of `queries/count_genai_total.sql`.

In [8]:
COV_PATH = PROCESSED / "coverage_validation.parquet"

total = pd.read_csv(RAW / "monthly_genai_total.csv")
total["month"] = total["month"].astype(str).apply(lambda x: pd.Period(x, freq="M"))
total = total.set_index("month")["total_articles"].rename("total_genai")

cov = vol.rename("gov_articles").to_frame().join(total, how="outer").fillna(0)
cov["coverage_rate"] = cov["gov_articles"] / cov["total_genai"]

# Save with string index (Period not parquet-native)
cov.index = cov.index.astype(str)
cov.to_parquet(COV_PATH)

overall_rate = cov["gov_articles"].sum() / cov["total_genai"].sum()
print(f"Governance coverage rate: {overall_rate:.1%} of all GenAI articles carry a governance signal")
cov.tail()

Governance coverage rate: 50.9% of all GenAI articles carry a governance signal


,gov_articles,total_genai,coverage_rate
month,,,
2026-02,27052,51043,0.529985
2026-03,31145,53366,0.583611
2026-04,27568,49833,0.553208
2026-05,28582,52498,0.544440
2026-06,20680,36034,0.573902


## Section 6 — Frame coverage statistics

In [9]:
from src.analysis import frame_coverage_rate

STATS_PATH = PROCESSED / "coverage_stats.json"

stats = frame_coverage_rate(df)
STATS_PATH.write_text(json.dumps(stats, indent=2))

print(f"Total articles:      {stats['total']:,}")
print(f"No frame matched:    {stats['zero_frame_pct']:.1%}")
print(f"≥1 frame matched:    {stats['any_frame_pct']:.1%}")
print(f"≥2 frames matched:   {stats['multi_frame_pct']:.1%}")
print(f"\nSaved → {STATS_PATH.relative_to(ROOT)}")

Total articles:      1,116,091
No frame matched:    49.4%
≥1 frame matched:    50.6%
≥2 frames matched:   17.4%

Saved → data/processed/coverage_stats.json


## Section 7 — Tone aggregates

GDELT V2Tone is a coarse heuristic; treat results directionally, not as ground-truth sentiment.
- `tone` = positive% − negative% (positive values = more positive coverage)
- Aggregated here by month, dominant frame, and region for use in supplementary figures.

In [10]:
from src.analysis import tone_over_time, tone_by_frame, tone_by_region

TONE_M_PATH = PROCESSED / "tone_monthly.parquet"
TONE_F_PATH = PROCESSED / "tone_by_frame.parquet"
TONE_R_PATH = PROCESSED / "tone_by_region.parquet"

# Monthly trend
tone_monthly = tone_over_time(df).rename("tone")
tone_monthly.index = tone_monthly.index.astype(str)
tone_monthly.to_frame().to_parquet(TONE_M_PATH)

# By dominant frame
tone_frame = tone_by_frame(df)
tone_frame.to_parquet(TONE_F_PATH)

# By region
tone_region = tone_by_region(df)
tone_region.to_parquet(TONE_R_PATH)

print("Tone by dominant frame (mean ± std, sorted most → least negative):")
tone_frame.sort_values("mean")

Tone by dominant frame (mean ± std, sorted most → least negative):


,mean,std,count
dominant_frame,,,
risk_safety,-2.301971,3.576997,89115
regulation_governance,-1.836077,3.349800,236809
misinformation_integrity,-1.143251,2.698997,31815
rights_privacy,-1.011717,3.607668,24881
economic_competition_labour,0.134665,3.208949,81797
innovation_opportunity,1.114535,3.754897,100505
